# HLLSet Algebra — Advanced Programming

Low-level Rust API of hllset-next: TF vector, Commit chain, rank algebra,
temporal pyramid, universal bridge, namespace prefixes, and IICA properties.

**New in this edition:** §9 Temporal Pyramid, §10 Universal Bridge, §11 Namespace Prefixes.

In [ ]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }
:dep hllset-ranks = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-ranks" }
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }
:dep hllset-temporal = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-temporal" }
:dep hllset-bridge = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-bridge" }

use hllset_core::*;
use hllset_core::core::tfvec::TFVec;
use hllset_core::core::commit::Commit;
use hllset_core::core::content_addr;
use hllset_ranks::*;
use hllset_ranks::register::*;
use hllset_ranks::fisher::*;
use hllset_ranks::mask::*;
use hllset_ranks::hllset::*;
use hllset_dsl::LatticeElement;
use hllset_temporal::TemporalPyramid;
use hllset_bridge::{re_represent, extract_3gram, spearman_rank_correlation, bridge};
use std::collections::HashMap;

println!("All crates loaded — M={M}, TOTAL_BITS={}", hllset_core::core::hllset::TOTAL_BITS);

---
## 1. TFVec — Bit-Level Term Frequency

Monotonic CRDT: values only increase. Wire format: 4-byte LE + 32,768×f64 = 262,148 bytes.

In [ ]:
let mut tf = TFVec::new();
tf.increment(0, 1.0); tf.increment(100, 5.0); tf.increment(0, 2.0);
println!("TF[0]: {:.1}, TF[100]: {:.1}, Total: {:.1}", tf.get(0), tf.get(100), tf.total());

// CRDT merge: pointwise maximum
let mut a = TFVec::new(); a.increment(0, 10.0); a.increment(1, 5.0);
let mut b = TFVec::new(); b.increment(0, 7.0);  b.increment(1, 12.0);
a.merge(&b);
println!("Merge: TF[0]={:.1}, TF[1]={:.1} (takes max)", a.get(0), a.get(1));

// Wire format roundtrip
let bytes = tf.to_bytes();
println!("Serialized: {} bytes, roundtrip OK: {}", bytes.len(),
         tf.values == TFVec::from_bytes(&bytes).unwrap().values);

---
## 2. Commit — D/R/N Lattice Evolution

Each evolution step produces a Commit (D/R/N). Content-addressed chain via `t:<sha1>`.

In [ ]:
let prev = LatticeElement::from_tokens(&["a", "b", "c", "d"]);
let curr = LatticeElement::from_tokens(&["b", "c", "d", "e"]);
let r = prev.intersection(&curr);
let d = prev.difference(&curr);
let n = curr.difference(&prev);
println!("R: {} bits, D: {} bits, N: {} bits", r.popcount(), d.popcount(), n.popcount());

let commit = Commit::new(curr.key(), "t:prev_head", d.key(), r.key(), n.key());
println!("Key: {}", commit.content_key());
println!("Chain valid: {}", commit.chain_valid("t:prev_head"));

---
## 3. Five-Level Rank Algebra

| Level | Function | What it ranks |
|-------|----------|---------------|
| 1 | F(TF) | Token frequency |
| 2 | G({token-R}) | Bit position |
| 3 | H({bit-R}) | Register (32 bits) |
| 4 | K(degree) | HLLSet |
| 5 | L(max)/M(min) | Compound |

See `09_rank_algebra.ipynb`. Below: the TF→Register bridge.

---
## 4. Per-Register TF Ranking (TfRegisterRanker)

Bridges 32,768-entry TF vector to 1,024 register-level ranks — no TokenLUT needed.

In [ ]:
let mut tf = TFVec::new();
for pos in 0..8 { tf.increment(pos, 100.0); }
for pos in 32..40 { tf.increment(pos, 50.0); }
for pos in 64..68 { tf.increment(pos, 10.0); }

let ranker = TfRegisterRanker::default();
let reg0 = ranker.rank_register(&tf, 0);
let reg1 = ranker.rank_register(&tf, 1);
let reg2 = ranker.rank_register(&tf, 2);
println!("Reg 0: rank={}, slots={}", reg0.value, reg0.active_slots);
println!("Reg 1: rank={}, slots={}", reg1.value, reg1.active_slots);
println!("Reg 2: rank={}, slots={}", reg2.value, reg2.active_slots);
assert!(reg0.value > reg1.value && reg1.value > reg2.value);

let top5 = ranker.top_k(&tf, 5);
println!("Top-5: {:?}", top5.iter().map(|r| (r.register, r.value)).collect::<Vec<_>>());

---
## 5. Noether Controller (Integer Flux)

Integer-only drift detection with halving decay (replaces float 0.9).

In [ ]:
struct FluxMonitor { flux: i64, threshold: i64 }
impl FluxMonitor {
    fn new(t: i64) -> Self { Self { flux: 0, threshold: t } }
    fn record_new(&mut self) { self.flux += 1; }
    fn record_evict(&mut self) { self.flux -= 1; }
    fn tick(&mut self) -> bool { let d = self.flux.abs() > self.threshold; self.flux /= 2; d }
}
let mut m = FluxMonitor::new(5);
for _ in 0..10 { m.record_new(); }
println!("10 inserts: flux={}", m.flux);
println!("Tick: flux={} drift={}", m.flux, m.tick());
for _ in 0..5 { m.tick(); }
assert_eq!(m.flux, 0);
println!("Decayed to zero — no floating point");

---
## 6. Fisher Matrix — Cross-Layer Bit Coupling

In [ ]:
let mut fisher = FisherMatrix::new();
fisher.add_layer(&LatticeElement::from_tokens(&["urgent", "now", "alert"]));
fisher.add_layer(&LatticeElement::from_tokens(&["urgent", "later", "review"]));
fisher.add_layer(&LatticeElement::from_tokens(&["archive", "later", "done"]));
println!("Layers: {}, entries: {}", fisher.layer_count(), fisher.entry_count());
println!("Most persistent: {:?}", fisher.most_persistent(3));

---
## 7. Observable Mask — Rank Depletion

Controls attention via rank threshold. Every HLLSet remains retrievable by key.

In [ ]:
let mut idx = HLLSetRankIndex::new();
// Insert ranks with explicit values: key, degree, popcount, rank-fn
for (key, degree, popcount) in [("h:hot", 5usize, 100u64), ("h:warm", 4, 80),
    ("h:cool", 3, 60), ("h:cold", 2, 40), ("h:frozen", 1, 20)] {
    let r = HLLSetRank::from_raw(key, degree, popcount, &DegreeRankFn);
    idx.insert(r);
}
println!("Total: {}", idx.len());
let mask50 = ObservableMask::apply(&idx, 50);
let mask90 = ObservableMask::apply(&idx, 90);
println!("At theta=50: {}/{} observable", mask50.observable_count(), mask50.total);
println!("At theta=90: {}/{} observable", mask90.observable_count(), mask90.total);
assert!(mask50.observable_count() > mask90.observable_count());

---
## 8. IICA Properties

Idempotency, Immutability, Content-Addressability — compose to guarantee convergence.

In [ ]:
let h1 = HLLSet::from_tokens(&["alice", "bob", "carol"]);
let h2 = HLLSet::from_tokens(&["alice", "bob", "carol"]);
assert_eq!(h1.content_key(), h2.content_key());
assert_eq!(h1.union(&h1).popcount(), h1.popcount());
println!("IICA verified — key: {}", h1.content_key());

---
## 9. Temporal Pyramid — L0–L6 Sliding Windows

The `hllset-temporal` crate implements configurable N-layer temporal pyramids
with automatic carry cascade at time boundaries (§4.2 of STANDARD.md).

```text
Layer 0  SECOND   L0 = ∪ S(t) over 1 second     ← finest
Layer 1  MINUTE   L1 = ∪ S(t) over 60 seconds
...cascade upward...
Layer 6  YEAR     L6 = ∪ S(t) over 365 days      ← coarsest
```

At each boundary: data carries upward (`L_{i+1} = L_{i+1} ∪ L_i; L_i = ∅`).
System state H_system = L0 ∪ L1 ∪ ... ∪ L6 — bit-lossless union.

In [ ]:
use std::time::Duration;

// Minimal 2-layer pyramid: 1s + 60s windows
let mut p = TemporalPyramid::minimal();
println!("Layers: {}", p.layer_count());

// Ingest observations with explicit time deltas (for reproducibility)
let obs_a = HLLSet::from_tokens(&["hello", "world"]);
let obs_b = HLLSet::from_tokens(&["hello", "rust"]);

p.ingest_with_delta(&obs_a, Duration::from_millis(500));
println!("After 0.5s: L0.obs={}, L1.obs={}", p.layer(0).observations, p.layer(1).observations);

// Cross 1-second boundary → L0 carries to L1
let carries = p.ingest_with_delta(&obs_b, Duration::from_secs(1));
println!("After 1.5s: {} carries, L0.obs={}, L1.obs={}", carries,
         p.layer(0).observations, p.layer(1).observations);

### System state is the union of all layers

In [ ]:
let state = p.system_state();
println!("System state popcount: {}", state.popcount());
println!("Total observations: {}", p.total_observations());

// TF snapshot from the current system state
let snapshot = p.tf_snapshot();
println!("TF snapshot total: {:.1}", snapshot.total());

// Per-layer snapshots
let per_layer = p.per_layer_tf_snapshots();
for (i, tf) in per_layer.iter().enumerate() {
    println!("  L{}: TF total={:.1}", i, tf.total());
}

### Configurable pyramids

The pyramid shape is fully tunable — same operations at any scale.

In [ ]:
// High-frequency trading: 5 × 100ms layers
let hft = TemporalPyramid::high_frequency();
println!("HFT pyramid: {} layers", hft.layer_count());

// Standard 7-layer (second → year)
let std = TemporalPyramid::standard();
println!("Standard pyramid: {} layers", std.layer_count());

// Custom: 3 layers (1s, 5s, 10s)
let custom = TemporalPyramid::new(vec![
    Duration::from_secs(1), Duration::from_secs(5), Duration::from_secs(10)
]);
println!("Custom pyramid: {} layers", custom.layer_count());

// Noether invariant holds
println!("Noether invariant: {}", p.verify_noether());

---
## 10. Universal Bridge — Cross-Domain Re-Representation

The `hllset-bridge` crate implements two-pass ingestion (§5 of STANDARD.md):

**Pass 1 (Representation):** `domain_input → murmurhash3 → H_src`
**Pass 2 (Re-Representation):** `H_src bits → "reg:{r}:tz:{tz}" tokens → H_bridge`

H_bridge lives in the target bit space — BSS, R-links, union/intersection
all work directly. The bridge transfers **structure**, not statistics.

In [ ]:
// Pass 1: create source HLLSet
let src = HLLSet::from_tokens(&["red", "car", "intersection"]);
println!("Source: {} bits, key: {}", src.popcount(), src.content_key());

// Pass 2: re-represent into target bit space
let bridge_hllset = re_represent(&src);
println!("Bridge: {} bits, key: {}", bridge_hllset.popcount(), bridge_hllset.content_key());

// Idempotent: same source → same bridge
let bridge2 = re_represent(&src);
assert_eq!(bridge_hllset.content_key(), bridge2.content_key());
println!("Re-representation is idempotent (IICA)");

### 3-gram Structural Fingerprinting

Encodes adjacency patterns AND vocabulary — the structural invariant
for cross-domain matching.

In [ ]:
let tokens = ["the", "cat", "sat", "on", "the", "mat"];
let fp = extract_3gram(&tokens);
println!("3-gram fingerprint: {} bits", fp.popcount());

// Deterministic: same tokens → same fingerprint
let fp2 = extract_3gram(&tokens);
assert_eq!(fp.content_key(), fp2.content_key());
println!("3-gram fingerprinting is deterministic");

### Spearman Rank Correlation

Ranks vectors by value order, then computes Pearson correlation on ranks.
ρ = 1.0 = perfect agreement, ρ = -1.0 = perfect inverse.

In [ ]:
let a = vec![100u64, 80, 60, 40, 20];
let b = vec![100u64, 80, 60, 40, 20];
let c = vec![20u64, 40, 60, 80, 100];

println!("ρ(identical): {:.3}", spearman_rank_correlation(&a, &b));
println!("ρ(inverse):   {:.3}", spearman_rank_correlation(&a, &c));

### Full Bridge Pipeline

Re-represent + 3-gram fingerprint + Spearman rank correlation against
a target lattice of candidate HLLSets.

In [ ]:
let src = HLLSet::from_tokens(&["red", "car", "intersection"]);
let rule = HLLSet::from_tokens(&["slow", "down", "intersection"]);
let unrelated = HLLSet::from_tokens(&["weather", "report", "sunny"]);

let mut candidates = HashMap::new();
candidates.insert("rule".to_string(), rule);
candidates.insert("unrelated".to_string(), unrelated);

let result = bridge(&src, &candidates, 2);
println!("Bridge: {} bits", result.bridge.popcount());
for (key, rho) in &result.matches {
    println!("  {key}: ρ={rho:.3}");
}

---
## 11. Namespace Prefixes — Full CID Taxonomy

STANDARD.md §2.2 defines 10 content-addressed prefixes + 1 temporal namespace.
The `content_addr` module now supports the complete taxonomy.

In [ ]:
use hllset_core::core::content_addr::*;

// All 10 valid CA prefixes
println!("Valid CA prefixes:");
for p in VALID_PREFIXES {
    println!("  {p}: — {}", is_valid_prefix(p));
}
println!("  x: — {} (invalid)", is_valid_prefix("x"));

### make_cid / parse_cid

Construct and decompose content-addressed identifiers.

In [ ]:
let sha1 = "a3f82c1d00000000000000000000000000000000";
let cid = make_cid("h", sha1).unwrap();
println!("make: {cid}");

let (prefix, hash) = parse_cid(&cid).unwrap();
println!("parse: prefix={prefix}, sha1={hash}");

// Invalid prefix rejected
assert!(make_cid("x", sha1).is_none());
assert!(parse_cid("h:too_short").is_none());

### Per-prefix generators

Each namespace has a dedicated key generator.

In [ ]:
let tokens: &[&[u8]] = &[b"hello", b"world"];

println!("Original (o):  {}", original_key_from_tokens(tokens));
println!("HLLSet  (h):   {}", content_key_from_tokens(tokens));
println!("Retained (r):  {}", retained_key_from_tokens(tokens));
println!("Departed (d):  {}", departed_key_from_tokens(tokens));
println!("New      (n):  {}", new_key_from_tokens(tokens));
println!("View     (v):  {}", view_key_from_tokens(tokens));
println!("LLM ctx  (l):  {}", llm_context_key_from_tokens(tokens));
println!("Commit   (t):  {}", commit_key_from_bytes(b"{\"ts\":1}"));

// User-assigned (UUID format)
let uk = user_key_from_uuid("550e8400-e29b-41d4-a716-446655440000").unwrap();
println!("User     (u):  {uk}");

// System keys
println!("System:        {}", system_key("tf"));
println!("System:        {}", system_key("head"));

### D/R/N decomposition with correct prefixes

Using the right namespace for each component of lattice evolution.

In [ ]:
let prev = LatticeElement::from_tokens(&["a", "b", "c"]);
let curr = LatticeElement::from_tokens(&["b", "c", "d"]);

// Map each component to its proper namespace
println!("Source (h): {}", prev.key());
println!("R-link  (r): h:...{} (intersection)", &prev.intersection(&curr).key()[2..12]);
println!("Departed (d): h:...{} (prev \\ curr)", &prev.difference(&curr).key()[2..12]);
println!("New      (n): h:...{} (curr \\ prev)", &curr.difference(&prev).key()[2..12]);

---
## Summary

| Component | Key property |
|-----------|-------------|
| **TFVec** | Monotonic CRDT, 262KB wire format |
| **Commit** | D/R/N content-addressed chain |
| **Rank algebra** | Integer-only, FPGA-native |
| **TfRegisterRanker** | TF → register rank, no LUT |
| **Noether** | Integer flux, halving decay |
| **Fisher** | Cross-layer bit co-occurrence |
| **Observable mask** | Rank-threshold attention filter |
| **Temporal pyramid** | L0–L6 carry cascade, configurable N |
| **Universal bridge** | Two-pass re-representation, Spearman ρ |
| **Namespace prefixes** | o/r/d/n/t/v/l + u/system: CID taxonomy |

All built on IICA: idempotent, immutable, content-addressed.